# Camanchaca-Predict · Pipeline completo en un solo notebook (Run All)

**G5 — Proyecto Aplicado 2026-2 (PUCV)** · Estimación de la **visibilidad en metros**
bajo camanchaca para el **Camino La Pólvora (Valparaíso)**: ResNet-50 (baseline) vs
**ViT-B/16 (protagonista)**.

Este notebook une el flujo completo de los notebooks 00–06 en una sola ejecución:

| Sección | Etapa | Produce |
|---|---|---|
| 1 | Datos: RESIDE vía Kaggle **o modo demo automático** | `reside_manifest.csv` |
| 2 | Etiquetado físico (Koschmieder, V = 3.912/β) + validación | `labels.csv` |
| 3 | EDA + split 70/15/15 anti-fuga por escena | `splits/*.csv`, MU/SIGMA |
| 4 | Baseline ResNet-50 (regresión log10 V) | checkpoint + métricas |
| 5 | Protagonista ViT-B/16 dual (regresión + 4 bandas) | checkpoint + métricas |
| 6 | Comparación final + figuras para los slides | `resumen_para_slides.md` |

## Cómo ejecutar
1. `Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4`.
2. `Entorno de ejecución → Ejecutar todo` (Ctrl+F9) y autoriza Google Drive.

**Sin `kaggle.json` el notebook NO falla:** cae automáticamente al modo demo
(escenas sintéticas + niebla ASM con β conocido) y ejecuta el pipeline completo
de punta a punta. **`FAST_RUN = True`** (celda de configuración, abajo) acorta los
entrenamientos para la demo (~15–25 min en T4); con `False` entrena con épocas
completas sobre RESIDE real.


In [ ]:
# Verificamos GPU disponible
import shutil, subprocess
if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
else:
    print("Sin GPU: activa T4 en Entorno de ejecución → Cambiar tipo de entorno")

In [ ]:
# Dependencias que no vienen preinstaladas en Colab (torch/torchvision ya están)
!pip install -q timm kaggle
print("Dependencias instaladas")

In [ ]:
# Setup global único (equivale al "setup estándar" de todos los notebooks 00-06)
import json, math, random, re, shutil, subprocess, sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
import timm

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Google Drive (persistencia entre sesiones) + carpetas del proyecto
try:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/camanchaca")
except ImportError:
    ROOT = Path("local_workspace")

DATA_DIR = ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
MANIFEST_PATH = DATA_DIR / "processed" / "reside_manifest.csv"
LABELS_PATH = DATA_DIR / "processed" / "labels.csv"
SPLITS_DIR = DATA_DIR / "splits"
CKPT_DIR = ROOT / "checkpoints"
RESULTS_DIR = ROOT / "results"
FIG_DIR = ROOT / "figures"
for p in (RAW_DIR, MANIFEST_PATH.parent, SPLITS_DIR, CKPT_DIR, RESULTS_DIR, FIG_DIR):
    p.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})
print(f"Raíz de datos: {ROOT}")

In [ ]:
# Versiones del entorno (registrar en el reporte para reproducibilidad)
import numpy, pandas, sklearn, matplotlib, PIL, timm
print("numpy     ", numpy.__version__)
print("pandas    ", pandas.__version__)
print("sklearn   ", sklearn.__version__)
print("torch     ", torch.__version__)
print("torchvision", __import__("torchvision").__version__)
print("timm      ", timm.__version__)

In [ ]:
# ══════════ CONFIGURACIÓN GLOBAL DEL PIPELINE ══════════
# FAST_RUN = True  -> demo ágil: modo demo sintético + épocas reducidas (~15-25 min T4)
# FAST_RUN = False -> run real: RESIDE (requiere kaggle.json + DATASET_SLUG) + épocas completas
FAST_RUN = True

# Slug de Kaggle verificado que contenga RESIDE-OTS (p. ej. "usuario/reside-ots").
# Vacío = el pipeline continúa en MODO DEMO (síntesis ASM con beta conocido).
DATASET_SLUG = ""

print(f"FAST_RUN = {FAST_RUN}  |  DATASET_SLUG = {DATASET_SLUG or '(vacío → modo demo)'}")

# Sección 1 · Datos — RESIDE desde Kaggle (o modo demo automático)

Descarga **RESIDE-OTS** con tus credenciales de Kaggle. Si no hay `kaggle.json`
o `DATASET_SLUG`, el pipeline **no se detiene**: genera automáticamente escenas
limpias (Plan C) y las VERSIONES con niebla sintética (modelo ASM con β conocido).

**Salida:** `data/processed/reside_manifest.csv` — ruta, escena, A y β por imagen.

In [ ]:
# Credenciales de Kaggle (OPCIONAL, no bloquea el Run All): busca kaggle.json
# en Drive. Si no existe, el pipeline continúa en MODO DEMO sin detenerse.
import shutil
from pathlib import Path

KAGGLE_OK = False
for cand in [Path("/content/drive/MyDrive/kaggle.json"),
             Path("/content/kaggle.json"),
             Path.home() / ".kaggle" / "kaggle.json"]:
    if cand.exists():
        kg = Path.home() / ".kaggle"
        kg.mkdir(exist_ok=True)
        shutil.copy(cand, kg / "kaggle.json")
        (kg / "kaggle.json").chmod(0o600)
        KAGGLE_OK = True
        print("Credenciales de Kaggle listas:", cand)
        break

if not KAGGLE_OK:
    print("Sin kaggle.json → no se descargará RESIDE desde Kaggle.")
    print("El pipeline usará el MODO DEMO (síntesis ASM controlada).")
    print("Para datos reales: guarda kaggle.json en la raíz de Drive y")
    print("re-ejecuta (kaggle.com → Account → Create New API Token).")

In [ ]:
# BÚSQUEDA (informativa, solo con credenciales): espejos de RESIDE en Kaggle
if KAGGLE_OK:
    r = subprocess.run(["kaggle", "datasets", "list", "-s", "reside",
                        "--sort-by", "votes"], capture_output=True, text=True)
    lines = (r.stdout or "").splitlines()
    print("\n".join(lines[:20]) if lines else "Sin resultados")
else:
    print("Búsqueda omitida (sin credenciales de Kaggle) → modo demo.")

De la lista anterior elige un espejo que contenga **RESIDE-OTS** (o
RESIDE-Standard con carpeta OTS). Verifica en kaggle.com la descripción antes
de descargar. Luego pega el slug (formato `usuario/dataset`) en la celda
siguiente.

In [ ]:
# DESCARGA: solo si hay credenciales Y slug verificado (ver celda anterior)
if KAGGLE_OK and DATASET_SLUG:
    print(f"Descargando {DATASET_SLUG} (puede tardar varios minutos)...")
    r = subprocess.run(["kaggle", "datasets", "download", "-d", DATASET_SLUG,
                        "-p", str(RAW_DIR), "--unzip"],
                       capture_output=True, text=True)
    print(r.stdout[-2000:] if r.stdout else "")
    print(r.stderr[-2000:] if r.returncode else "Descarga completada")
else:
    print("Descarga omitida (sin credenciales o sin DATASET_SLUG).")
    print("→ El pipeline continuará en MODO DEMO (síntesis ASM controlada).")

In [ ]:
# Inspeccionamos la estructura descargada (carpetas y tamaños)
def show_tree(root, depth=2, prefix=""):
    root = Path(root)
    entries = sorted(root.iterdir(), key=lambda p: (p.is_file(), p.name))
    for i, p in enumerate(entries):
        last = i == len(entries) - 1
        print(prefix + ("└── " if last else "├── ") + p.name +
              ("/" if p.is_dir() else ""))
        if p.is_dir() and depth > 1:
            show_tree(p, depth - 1, prefix + ("    " if last else "│   "))

show_tree(RAW_DIR, depth=2)

In [ ]:
# Detección automática de imágenes RESIDE con niebla (patrón escena_A_beta)
FILENAME_RE = re.compile(r"^(?P<scene>\d+)_(?P<atmlight>[\d.]+)_(?P<beta>[\d.]+)$")

def find_reside_hazy(root, path_filter=""):
    # Encuentra imágenes cuyo nombre sigue el patrón escena_A_beta.
    # path_filter: substring obligatorio en la ruta (p.ej. 'OTS' para solo outdoor).
    hits = []
    for ext in ("*.jpg", "*.jpeg", "*.png"):
        for p in Path(root).rglob(ext):
            if FILENAME_RE.match(p.stem) and (not path_filter or path_filter.lower() in str(p).lower()):
                hits.append(p)
    return sorted(set(hits))

# Si el espejo mezcla indoor (ITS) y outdoor (OTS), filtra por 'OTS'.
PATH_FILTER = ""      # <-- cambia a "OTS" si es necesario
hazy_paths = find_reside_hazy(RAW_DIR, path_filter=PATH_FILTER)
print(f"Imágenes con niebla detectadas: {len(hazy_paths)}")
if hazy_paths:
    print("Ejemplos:", [p.name for p in hazy_paths[:5]])

In [ ]:
# Construcción del MANIFIESTO: ruta + escena + A + beta por imagen
def parse_stem(stem):
    m = FILENAME_RE.match(stem)
    if m is None:
        return None
    return {"scene": m.group("scene"),
            "A": float(m.group("atmlight")),
            "beta": float(m.group("beta"))}

records = []
for p in hazy_paths:
    info = parse_stem(p.stem)
    if info:
        records.append({"image": p.name, "path": str(p),
                        "scene": info["scene"], "A": info["A"],
                        "beta": info["beta"], "source": "reside"})

manifest = pd.DataFrame(records)
if len(manifest):
    print(f"Manifiesto: {len(manifest)} imágenes · {manifest['scene'].nunique()} escenas")
    print(f"Rango de beta: [{manifest['beta'].min():.4f}, {manifest['beta'].max():.4f}]")
    display(manifest.head())
else:
    print("⚠️ Sin imágenes RESIDE. Usa el modo DEMO de la celda siguiente.")

### Modo DEMO (solo si la descarga falló)

Para que el pipeline completo funcione aunque Kaggle falle (p. ej. durante la
presentación), este modo sintetiza niebla **con β conocido** sobre fotos
propias (sube ~20 fotos exteriores a `Drive/camanchaca/data/raw/demo_clear/`)
usando el modelo atmosférico de dispersión con profundidad plana aproximada.

> ⚠️ Las etiquetas del modo demo son válidas **por construcción** (nosotros
> elegimos β), pero las imágenes NO son representativas: sirve para probar el
> pipeline, no para reportar resultados.

In [ ]:
# PLAN C — escenas limpias automáticas (Run All sin Kaggle ni fotos propias):
# sintetizamos paisajes procedurales con estructura de profundidad
# (cielo con gradiente, siluetas de cerros, carretera) en demo_clear/.
import numpy as np
from PIL import Image, ImageDraw, ImageFilter

DEMO_CLEAR = RAW_DIR / "demo_clear"
DEMO_HAZY = RAW_DIR / "demo_hazy"
DEMO_CLEAR.mkdir(parents=True, exist_ok=True)
DEMO_HAZY.mkdir(parents=True, exist_ok=True)

_clear_existing = sorted(DEMO_CLEAR.glob("*.jpg"))

if len(manifest) == 0 and len(_clear_existing) == 0:
    rng = np.random.RandomState(SEED)
    W, H = 320, 240
    for s in range(12):
        # cielo con gradiente vertical (tono aleatorio suave por escena)
        top = rng.uniform(0.45, 0.75, 3)
        bottom = rng.uniform(0.75, 0.95, 3)
        sky = np.linspace(0, 1, H // 2)[:, None, None]
        sky_rgb = top[None, None] * (1 - sky) + bottom[None, None] * sky
        # suelo con gradiente hacia el horizonte
        gnd = np.linspace(1, 0.55, H - H // 2)[:, None, None]
        gnd_rgb = np.array([0.32, 0.38, 0.30])[None, None] * gnd
        img = np.vstack([sky_rgb, gnd_rgb])
        img = np.repeat(img, W, axis=1)
        pim = Image.fromarray((np.clip(img, 0, 1) * 255).astype(np.uint8))
        draw = ImageDraw.Draw(pim)
        horizon = H // 2 + int(rng.randint(-8, 8))
        # 2-3 siluetas de cerros: más lejos = más claro (pista de profundidad)
        for k in range(rng.randint(2, 4)):
            base = 150 + 35 * k
            col = tuple(min(255, int(base * c)) for c in (1.0, 1.0, 1.05))
            pts, x = [(0, H)], 0
            y0 = horizon - rng.randint(5, max(6, 40 - 8 * k))
            while x <= W:
                pts.append((x, y0 + int(rng.randint(-14, 14))))
                x += rng.randint(24, 56)
            pts += [(W, H), (0, H)]
            draw.polygon(pts, fill=col)
        # "carretera" trapezoide: se estrecha hacia el horizonte
        draw.polygon([(W // 2 - 14, H), (W // 2 + 14, H),
                      (W // 2 + 55, horizon + 8), (W // 2 - 55, horizon + 8)],
                     fill=(72, 74, 78))
        # postes laterales crecientes: textura de escala/depth
        for i in range(4):
            xw = int(W * (0.12 + 0.18 * i) + rng.randint(-6, 6))
            hgt = int(H * (0.16 + 0.07 * i))
            draw.rectangle([xw, horizon + 10 + i * 12,
                            xw + 5, horizon + 10 + i * 12 + hgt], fill=(48, 44, 40))
        pim = pim.filter(ImageFilter.GaussianBlur(0.6))
        pim.save(DEMO_CLEAR / f"scene_{s:02d}.jpg", quality=92)
    print(f"PLAN C: 12 escenas limpias sintéticas guardadas en {DEMO_CLEAR}")
elif len(manifest) == 0:
    print(f"PLAN C omitido: ya hay fotos propias en {DEMO_CLEAR} (se usarán)")
else:
    print("PLAN C omitido: hay manifiesto RESIDE disponible")

In [ ]:
# MODO DEMO: síntesis ASM con beta controlado (profundidad plana aproximada)
import numpy as np
from PIL import Image

DEMO_CLEAR = RAW_DIR / "demo_clear"
DEMO_HAZY = RAW_DIR / "demo_hazy"
DEMO_CLEAR.mkdir(parents=True, exist_ok=True)
DEMO_HAZY.mkdir(parents=True, exist_ok=True)

clear_imgs = sorted([p for p in DEMO_CLEAR.glob("*.*")
                     if p.suffix.lower() in {".jpg", ".jpeg", ".png"}])

if len(manifest) == 0 and len(clear_imgs) > 0:
    rng = np.random.RandomState(SEED)
    betas = [0.01, 0.02, 0.04, 0.08, 0.12, 0.2]   # V = 3.912/beta: 391..20 m (4 bandas)
    records = []
    for p in clear_imgs:
        img = np.asarray(Image.open(p).convert("RGB"), dtype=np.float64) / 255.0
        h, w = img.shape[:2]
        # profundidad relativa: de ~30 m (abajo/primero plano) a ~150 m (horizonte)
        depth = np.linspace(0.2, 1.0, h)[:, None] * np.ones((1, w)) * 150.0
        for k, beta in enumerate(rng.permutation(betas)[:3]):
            A = 0.85
            t = np.exp(-beta * depth)[..., None]
            hazy = np.clip(img * t + A * (1 - t), 0, 1)
            out = DEMO_HAZY / f"{p.stem}_{k:02d}_{A:.2f}_{beta:.3f}.jpg"
            Image.fromarray((hazy * 255).astype(np.uint8)).save(out, quality=92)
            records.append({"image": out.name, "path": str(out),
                            "scene": p.stem, "A": A, "beta": beta,
                            "source": "demo"})
    manifest = pd.DataFrame(records)
    print(f"DEMO: {len(manifest)} imágenes sintéticas con beta conocido")
elif len(manifest) == 0:
    print("Sin datos RESIDE ni fotos demo: sube fotos a", DEMO_CLEAR)
else:
    print("Manifiesto RESIDE disponible: modo demo omitido")

In [ ]:
# Subset controlado (Colab free): máximo ~12000 imágenes, proporcional por escena
MAX_IMAGES = 12000

if len(manifest) > MAX_IMAGES:
    frac = MAX_IMAGES / len(manifest)
    manifest = (manifest.groupby("scene", group_keys=False)
                        .sample(frac=frac, random_state=SEED)
                .reset_index(drop=True))
    print(f"Subset: {len(manifest)} imágenes ({manifest['scene'].nunique()} escenas)")
else:
    print(f"Se usan las {len(manifest)} imágenes disponibles")

# Verificación de legibilidad de una muestra
from PIL import Image
if len(manifest):
    for p in manifest["path"].sample(min(5, len(manifest)), random_state=SEED):
        im = Image.open(p); im.verify()
    print("Muestra de imágenes legible ✅")
else:
    print("⚠️ Manifiesto vacío: completa la descarga o el modo DEMO antes de continuar.")

In [ ]:
# Guardamos el manifiesto (Drive + local) — lo consume el notebook 02
import matplotlib.pyplot as plt

manifest.to_csv(MANIFEST_PATH, index=False)
print(f"Manifiesto guardado: {MANIFEST_PATH} ({len(manifest)} filas)")
if len(manifest):
    manifest["beta"].hist(bins=50, figsize=(7, 3))
    plt.title("Distribución de beta en el manifiesto")
    plt.xlabel("beta [1/m]"); plt.ylabel("imágenes"); plt.show()
else:
    print("⚠️ Manifiesto vacío: revisa la descarga (slug de Kaggle) o el modo DEMO.")

# Sección 2 · Etiquetado físico — de β a visibilidad en metros (Koschmieder)

Ningún dataset de niebla entrega visibilidad en metros: la etiqueta se **deriva
físicamente** con la ley de Koschmieder `V = −ln(0.02)/β ≈ 3.912/β`, se asigna la
**banda de seguridad vial** (justificada con la distancia de detención) y se
**valida** la consistencia con un proxy de niebla (canal oscuro, Spearman).

**Salida:** `labels.csv` (V en metros + banda por imagen).

### 1. Marco físico

#### Ley de Koschmieder
Un objeto oscuro a distancia `d` contra el cielo tiene contraste aparente
`C(d) = exp(-β·d)`. La **visibilidad** es la distancia donde el contraste cae
al umbral ε (usamos **ε = 0.02**, conservador para seguridad vial):

$$V = -\ln(\varepsilon)/\beta \;\approx\; 3.912/\beta \quad [\text{metros}]$$

#### Modelo atmosférico de dispersión (ASM)
RESIDE sintetiza niebla con `I(x) = J(x)·t(x) + A·(1−t(x))` y
`t(x) = exp(−β·d(x))`, registrando β en el nombre de archivo. Ese β
[1/m] es nuestra ancla física.

**Sensibilidad del umbral:** con ε = 0.05 (MOR de aviación), V = 3.0/β
(~23 % menor). Lo reportamos como análisis de sensibilidad, no cambiamos el
criterio principal.

In [ ]:
# Funciones del criterio de etiquetado (idénticas a src/camanchaca/labeling/)
DEFAULT_EPSILON = 0.02
BAND_EDGES = (50.0, 100.0, 200.0)
BAND_NAMES = ("critico", "alto_riesgo", "precaucion", "aceptable")
BAND_LABELS_ES = ("Crítico (<50 m)", "Alto riesgo (50–100 m)",
                  "Precaución (100–200 m)", "Aceptable (≥200 m)")

def visibility_from_beta(beta, epsilon=DEFAULT_EPSILON):
    beta = np.asarray(beta, dtype=float)
    return -np.log(epsilon) / beta

def band_from_visibility(v):
    v = np.asarray(v, dtype=float)
    return np.array(BAND_NAMES)[np.digitize(v, BAND_EDGES)]

def stopping_sight_distance(v_kmh, t_reaction=2.5, mu=0.4, g=9.81):
    v = np.asarray(v_kmh, dtype=float) / 3.6
    return v * t_reaction + v ** 2 / (2 * mu * g)

# Ejemplos numéricos de la conversión
ejemplos = pd.DataFrame({
    "beta_1_sobre_m": [0.078, 0.039, 0.02, 0.01, 0.005],
})
ejemplos["V_metros"] = visibility_from_beta(ejemplos["beta_1_sobre_m"]).round(1)
ejemplos["banda"] = [str(b) for b in band_from_visibility(ejemplos["V_metros"])]
ejemplos

In [ ]:
# Justificación de las bandas: distancia de detención (SSD) en piso mojado
ssd = pd.DataFrame({
    "velocidad_kmh": [40, 60, 80, 100, 120],
    "SSD_m": stopping_sight_distance([40, 60, 80, 100, 120]).round(1),
})
display(ssd)
print("Lectura: con V < 50 m ni frenando a 60 km/h se detiene a tiempo;")
print("100-200 m da margen hasta ~100 km/h; >=200 m incluso a 120 km/h.")

### 2. Construcción de labels.csv

Columnas generadas:

| Columna | Significado |
|---|---|
| `visibility_m` | etiqueta física V = 3.912/β |
| `log10_v` | target de regresión (estabiliza el rango ~10–2000 m) |
| `band`, `band_idx` | etiqueta de clasificación (0..3) |

In [ ]:
# Conversión β -> V -> banda para todo el manifiesto
labels = manifest.copy()
labels["visibility_m"] = visibility_from_beta(labels["beta"].values)
labels["log10_v"] = np.log10(labels["visibility_m"])
labels["band"] = [str(b) for b in band_from_visibility(labels["visibility_m"].values)]
labels["band_idx"] = np.digitize(labels["visibility_m"].values, BAND_EDGES)
labels["V_metros_eps05"] = -np.log(0.05) / labels["beta"]   # sensibilidad ε=0.05

display(labels[["image", "scene", "beta", "visibility_m", "log10_v", "band"]].head())
print(f"V: min={labels['visibility_m'].min():.1f} m · "
      f"mediana={labels['visibility_m'].median():.1f} m · "
      f"max={labels['visibility_m'].max():.1f} m")

In [ ]:
# Distribución de la visibilidad y balance de bandas
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
axes[0].hist(labels["visibility_m"], bins=60, color="#3681a6", edgecolor="white")
for e in BAND_EDGES:
    axes[0].axvline(e, color="crimson", ls="--", lw=1)
axes[0].set_xlabel("Visibilidad [m]"); axes[0].set_ylabel("imágenes")
axes[0].set_title("Distribución de V (cortes de banda en rojo)")

counts = labels["band"].value_counts().reindex(BAND_NAMES, fill_value=0)
axes[1].bar(BAND_LABELS_ES, counts.values, color=["#b2182b", "#ef8a62", "#67a9cf", "#2166ac"])
axes[1].set_title("Imágenes por banda de seguridad")
axes[1].tick_params(axis="x", rotation=20)
for i, c in enumerate(counts.values):
    axes[1].text(i, c, f"{c}\n({100*c/len(labels):.0f}%)", ha="center", va="bottom", fontsize=9)
plt.show()

print("⚠️ Si 'critico' queda < 5% -> activar mitigación R3 (class weights / re-síntesis).")

### 3. Validación de consistencia de las etiquetas

Las etiquetas dependen del β del nombre de archivo. **Validación perceptual:**
el canal oscuro (He et al. 2011) de una imagen con niebla densa es más
brillante (se acerca a la luz atmosférica). Si nuestro β es correcto, la
correlación de Spearman entre el proxy y β debe ser **alta y positiva**.

In [ ]:
# Proxy de niebla (canal oscuro) vs beta: correlación de Spearman
import cv2
from scipy.stats import spearmanr

def haze_density_proxy(img_bgr, patch=15):
    kernel = np.ones((patch, patch), np.uint8)
    dc = cv2.erode(img_bgr.min(axis=2), kernel)
    return float(dc.mean()) / 255.0

muestra = labels.sample(min(80, len(labels)), random_state=SEED)
proxies, betas = [], []
for _, row in muestra.iterrows():
    img = cv2.imread(row["path"])
    if img is None:
        continue
    proxies.append(haze_density_proxy(img))
    betas.append(row["beta"])

rho, pval = spearmanr(betas, proxies)
print(f"Spearman ρ(beta, proxy_canal_oscuro) = {rho:.3f}  (p = {pval:.2e}, n = {len(betas)})")

fig, ax = plt.subplots(figsize=(6, 4), constrained_layout=True)
ax.scatter(betas, proxies, s=18, alpha=0.7, color="#3681a6")
ax.set_xlabel("β del filename [1/m]"); ax.set_ylabel("proxy canal oscuro [0-1]")
ax.set_title(f"Consistencia perceptual (ρ = {rho:.2f})")
plt.show()

if rho > 0.7:
    print("✅ Etiquetas consistentes con la percepción de niebla.")
elif rho > 0.4:
    print("🟡 Correlación moderada: revisar subconjuntos / outliers (riesgo R2).")
else:
    print("🔴 Correlación débil: activar mitigación R2 (re-etiquetar por re-síntesis).")

**Validación física adicional (opcional, si se consigue ITS + depth maps):**
renderizar de nuevo la niebla con `t = exp(−β·d)` usando profundidad conocida
y comparar contra la imagen entregada confirma unidades y fórmula del
generador. Detalle en `docs/labeling_criteria.md` §5.

In [ ]:
# Guardamos labels.csv (Drive + copia local al repo si está clonado)
labels.to_csv(LABELS_PATH, index=False)
print(f"labels.csv guardado: {LABELS_PATH} ({len(labels)} filas)")

REPO_LOCAL = Path("/content/camanchaca-predict")
if REPO_LOCAL.exists():
    dest = REPO_LOCAL / "data" / "processed" / "labels.csv"
    dest.parent.mkdir(parents=True, exist_ok=True)
    labels.to_csv(dest, index=False)
    print(f"Copia local en el repo: {dest}")

### Resumen del notebook

- ✅ Etiqueta física por imagen: `V = 3.912/β` metros (+ sensibilidad ε=0.05).
- ✅ Bandas de seguridad con justificación SSD (50/100/200 m).
- ✅ Validación de consistencia perceptual (ρ de Spearman).
- ⚠️ Observar el balance de bandas → riesgo R3 si `critico` es minoritaria.

**Siguiente:** notebook 03 · eda_splits (EDA + split 70/15/15 agrupado por
escena, anti-fuga).

# Sección 3 · EDA y split train/val/test anti-fuga (70/15/15 por escena)

RESIDE genera varias versiones con niebla de la misma escena limpia: un split
aleatorio **filtraría información** (misma escena en train y test). El split es
**agrupado por escena**, buscando el mínimo sesgo de bandas entre conjuntos.

**Salida:** `splits/{train,val,test}.csv` + `class_weights.json` (MU/SIGMA + pesos).

In [ ]:
# Contexto del flujo continuo: `labels` ya está en memoria desde la Sección 2
# (no hace falta releer el CSV). Definimos el mapeo banda->índice para los
# class weights (equivalente al setup del notebook 03 individual).
BAND2IDX = {b: i for i, b in enumerate(BAND_NAMES)}
print(f"{len(labels)} imágenes · {labels['scene'].nunique()} escenas · "
      f"{labels['beta'].nunique()} betas distintos")

In [ ]:
# Chequeos básicos de calidad
print("Duplicados por (scene, beta):", labels.duplicated(subset=["scene", "beta"]).sum())
print("NaN:", labels.isna().sum().sum())
print("Rutas existentes:", labels["path"].map(lambda p: Path(p).exists()).mean())

tamanos = labels["path"].sample(min(100, len(labels)), random_state=SEED)\
                        .map(lambda p: Image.open(p).size)
print("Tamaños de imagen más comunes:", pd.Series(tamanos).value_counts().head(3).to_dict())
print("Decisión: resize+crop a 224x224 (notebooks 04/05).")

In [ ]:
# Galería: 4 ejemplos por banda (con su V real)
n_por_banda = min(4, int(labels["band"].value_counts().min()))
fig, axes = plt.subplots(4, n_por_banda, figsize=(3.2 * n_por_banda, 12),
                         constrained_layout=True, squeeze=False)
for bi, band in enumerate(BAND_NAMES):
    sub = labels[labels["band"] == band]
    sub = sub.sample(min(n_por_banda, len(sub)), random_state=SEED)
    for k in range(n_por_banda):
        ax = axes[bi, k] if n_por_banda > 1 else axes[bi]
        if k < len(sub):
            r = sub.iloc[k]
            ax.imshow(Image.open(r["path"]))
            ax.set_title(f"{r['visibility_m']:.0f} m", fontsize=10)
        ax.axis("off")
    axes[bi, 0].text(-0.08, 0.5, band, transform=axes[bi, 0].transAxes,
                     rotation=90, va="center", ha="center", fontsize=11, fontweight="bold")
fig.suptitle("Ejemplos por banda de seguridad (V real en el título)", fontsize=13)
plt.show()

### Split agrupado por escena

La unidad de split es la **escena**, no la imagen. Entre 50 permutaciones
aleatorias de escenas elegimos la de menor sesgo en la distribución de
bandas. Implementación idéntica a `src/camanchaca/data/splits.py`
(verificada por `tests/test_splits.py`).

In [ ]:
# Split agrupado por escena (anti-fuga) con búsqueda de mínimo sesgo de bandas
def _assign_scenes(counts, order, n_total, test_frac, val_frac):
    test_s, val_s, train_s = [], [], []
    n_test = n_val = 0
    for s in order:
        c = counts[s]
        if n_test < test_frac * n_total:
            test_s.append(s); n_test += c
        elif n_val < val_frac * n_total:
            val_s.append(s); n_val += c
        else:
            train_s.append(s)
    return train_s, val_s, test_s

def grouped_split(df, val_frac=0.15, test_frac=0.15, n_seeds=50, seed=SEED):
    counts = df.groupby("scene").size().to_dict()
    n_total = len(df)
    rng_master = np.random.RandomState(seed)
    best = None
    for _ in range(n_seeds):
        rng = np.random.RandomState(rng_master.randint(0, 2**31 - 1))
        order = rng.permutation(list(counts.keys()))
        tr_s, va_s, te_s = _assign_scenes(counts, order, n_total, test_frac, val_frac)
        tr = df[df["scene"].isin(tr_s)]
        va = df[df["scene"].isin(va_s)]
        te = df[df["scene"].isin(te_s)]
        glob = df["band"].value_counts(normalize=True)
        worst = max(float((d["band"].value_counts(normalize=True)
                           .reindex(glob.index, fill_value=0.0) - glob).abs().max())
                    for d in (tr, va, te) if len(d) > 0)
        if best is None or worst < best[0]:
            best = (worst, tr, va, te)
    _, tr, va, te = best
    return (tr.reset_index(drop=True), va.reset_index(drop=True), te.reset_index(drop=True))

train_df, val_df, test_df = grouped_split(labels)
print(f"train={len(train_df)} ({len(train_df)/len(labels):.0%}) · "
      f"val={len(val_df)} ({len(val_df)/len(labels):.0%}) · "
      f"test={len(test_df)} ({len(test_df)/len(labels):.0%})")

In [ ]:
# VERIFICACIÓN ANTI-FUGA (debe dar 0 en las tres líneas)
print("Escenas train ∩ val :", len(set(train_df["scene"]) & set(val_df["scene"])))
print("Escenas train ∩ test:", len(set(train_df["scene"]) & set(test_df["scene"])))
print("Escenas val   ∩ test:", len(set(val_df["scene"]) & set(test_df["scene"])))
assert (set(train_df["scene"]) & set(test_df["scene"])) == set(), 'FUGA DETECTADA'

In [ ]:
# Balance de bandas por conjunto (debe ser similar entre barras)
fig, ax = plt.subplots(figsize=(9, 4), constrained_layout=True)
x = np.arange(len(BAND_NAMES)); w = 0.26
for off, (nombre, d) in enumerate([("train", train_df), ("val", val_df), ("test", test_df)]):
    props = d["band"].value_counts(normalize=True).reindex(BAND_NAMES, fill_value=0.0)
    ax.bar(x + (off - 1) * w, props.values, w, label=f"{nombre} (n={len(d)})")
ax.set_xticks(x); ax.set_xticklabels(BAND_NAMES)
ax.set_ylabel("proporción"); ax.set_title("Distribución de bandas por conjunto")
ax.legend()
plt.show()

In [ ]:
# Target de regresión: estandarización de log10(V) con estadísticas de TRAIN
MU = float(train_df["log10_v"].mean())
SIGMA = float(train_df["log10_v"].std())

for d in (train_df, val_df, test_df):
    d["y_norm"] = (d["log10_v"] - MU) / SIGMA
    d["band_idx"] = d["band"].map(BAND2IDX).astype(int)
print(f"MU={MU:.4f}  SIGMA={SIGMA:.4f}  (se reutilizan en notebooks 04-06)")

# Pesos de clase para la pérdida CE del ViT (mitigación R3: desbalance)
freq = train_df["band"].value_counts().reindex(BAND_NAMES).fillna(1)
class_weights = (len(train_df) / (len(BAND_NAMES) * freq)).round(3)
print("class_weights:", class_weights.to_dict())

In [ ]:
# Exportamos splits + estadísticas (Drive + repo local si existe)
import json

train_df.to_csv(SPLITS_DIR / "train.csv", index=False)
val_df.to_csv(SPLITS_DIR / "val.csv", index=False)
test_df.to_csv(SPLITS_DIR / "test.csv", index=False)
with open(SPLITS_DIR / "class_weights.json", "w") as f:
    json.dump({"mu": MU, "sigma": SIGMA,
               "class_weights": class_weights.to_dict()}, f, indent=2)

REPO_LOCAL = Path("/content/camanchaca-predict")
if REPO_LOCAL.exists():
    dest = REPO_LOCAL / "data" / "splits"
    dest.mkdir(parents=True, exist_ok=True)
    train_df.to_csv(dest / "train.csv", index=False)
    val_df.to_csv(dest / "val.csv", index=False)
    test_df.to_csv(dest / "test.csv", index=False)
    import shutil
    shutil.copy(SPLITS_DIR / "class_weights.json", dest / "class_weights.json")
    print("Splits copiados también al repo (se commitean: son CSV pequeños)")

print(f"Splits guardados en {SPLITS_DIR}")
display(train_df[["image", "scene", "beta", "visibility_m", "band", "y_norm"]].head())

### Resumen del notebook

- ✅ Split 70/15/15 **agrupado por escena**, semilla 42, mínimo sesgo de bandas.
- ✅ Verificación anti-fuga (0 escenas compartidas) — evidencia para la rúbrica.
- ✅ `y_norm` estandarizado con estadísticas de train (MU/SIGMA guardadas).
- ✅ `class_weights.json` para el ViT (mitigación R3).

**Siguiente:** notebook 04 · baseline_resnet50 (entrenamiento del baseline
exigido por la rúbrica).

# Sección 4 · Baseline — ResNet-50 (regresión de visibilidad)

Fine-tuning completo de ResNet-50 con cabeza de regresión sobre `log10(V)`,
early stopping y checkpoint por época. Con `FAST_RUN=True` entrena 2 épocas
(solo para verificar el flujo y la demo); con `False`, 8 épocas.

**Salida:** `checkpoints/resnet50_best.pt` + `results/resnet50_metrics.json`.

In [ ]:
# Cargamos splits y estadísticas del target (notebook 03)
train_df = pd.read_csv(DATA_DIR / "splits" / "train.csv")
val_df = pd.read_csv(DATA_DIR / "splits" / "val.csv")
test_df = pd.read_csv(DATA_DIR / "splits" / "test.csv")
with open(DATA_DIR / "splits" / "class_weights.json") as f:
    stats = json.load(f)
MU, SIGMA = stats["mu"], stats["sigma"]
print(f"train={len(train_df)} val={len(val_df)} test={len(test_df)} · MU={MU:.3f} SIGMA={SIGMA:.3f}")

In [ ]:
# Dataset y transformaciones (sin jitter de color: alteraría la niebla)
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

train_tf = transforms.Compose([
    transforms.Resize(256), transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(), transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
eval_tf = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(224),
    transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])

class VisibilityDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        img = self.transform(Image.open(r["path"]).convert("RGB"))
        return (img, float(r["y_norm"]), int(r["band_idx"]), float(r["visibility_m"]))

BATCH = 64
train_loader = DataLoader(VisibilityDataset(train_df, train_tf), batch_size=BATCH,
                          shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(VisibilityDataset(val_df, eval_tf), batch_size=BATCH,
                        num_workers=2, pin_memory=True)
test_loader = DataLoader(VisibilityDataset(test_df, eval_tf), batch_size=BATCH,
                         num_workers=2, pin_memory=True)
print(f"Loaders listos · batch={BATCH}")

In [ ]:
# Modelo: ResNet-50 + cabeza de regresión (idéntico a src/camanchaca/models/)
class ResNetVisibility(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        net = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2
                              if pretrained else None)
        net.fc = nn.Identity()
        self.backbone = net
        self.head = nn.Sequential(nn.Linear(2048, 256), nn.ReLU(inplace=True),
                                  nn.Dropout(0.2), nn.Linear(256, 1))
    def forward(self, x):
        return self.head(self.backbone(x)).squeeze(-1)

model = ResNetVisibility(pretrained=True).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"ResNet-50 · {n_params/1e6:.1f} M parámetros")

In [ ]:
# Métricas y funciones de época (log10 V -> metros)
BAND_EDGES = (50.0, 100.0, 200.0)
BAND_NAMES = ("critico", "alto_riesgo", "precaucion", "aceptable")

def to_meters(y_norm):
    return 10 ** (y_norm * SIGMA + MU)

def compute_metrics(v_true, v_pred):
    from sklearn.metrics import f1_score
    err = v_pred - v_true
    mae = float(np.abs(err).mean())
    rmse = float(np.sqrt((err ** 2).mean()))
    mape = float((np.abs(err) / np.clip(v_true, 1e-6, None)).mean() * 100)
    ss_res = float((err ** 2).sum())
    ss_tot = float(((v_true - v_true.mean()) ** 2).sum())
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else float("nan")
    b_true = np.digitize(v_true, BAND_EDGES)
    b_pred = np.digitize(v_pred, BAND_EDGES)
    f1 = float(f1_score(b_true, b_pred, average="macro", zero_division=0))
    peligro = v_true < 100.0
    falso_seguro = float((v_pred[peligro] >= 100.0).mean()) if peligro.any() else 0.0
    return {"mae_m": mae, "rmse_m": rmse, "mape_pct": mape, "r2": r2,
            "f1_macro": f1, "false_safe_rate": falso_seguro}

def run_epoch(model, loader, optimizer=None, scaler=None):
    training = optimizer is not None
    model.train() if training else model.eval()
    losses, preds, vtrues = [], [], []
    with torch.set_grad_enabled(training):
        for x, y_norm, band, v_true in loader:
            x, y_norm = x.to(DEVICE, non_blocking=True), y_norm.to(DEVICE, non_blocking=True)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16,
                                enabled=training and scaler is not None):
                out = model(x)
                loss = nn.functional.mse_loss(out, y_norm)
            if training:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimizer); scaler.update()
            losses.append(loss.item() * x.size(0))
            preds.append(out.float().cpu().numpy())
            vtrues.append(v_true.numpy())
    return (np.sum(losses) / len(loader.dataset),
            to_meters(np.concatenate(preds)),
            np.concatenate(vtrues))

In [ ]:
# Entrenamiento completo con early stopping y checkpoint por época
EPOCHS = 2 if FAST_RUN else 8
PATIENCE = 3

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.amp.GradScaler(enabled=DEVICE.type == "cuda")

history, best_mae, best_epoch, no_improve = [], float("inf"), -1, 0
t0 = time.time()
for epoch in range(1, EPOCHS + 1):
    tr_loss, _, _ = run_epoch(model, train_loader, optimizer, scaler)
    with torch.no_grad():
        va_loss, va_pred, va_true = run_epoch(model, val_loader)
    va_m = compute_metrics(va_true, va_pred)
    history.append({"epoch": epoch, "train_loss": tr_loss, "val_loss": va_loss, **va_m})
    marker = ""
    if va_m["mae_m"] < best_mae:
        best_mae, best_epoch, no_improve = va_m["mae_m"], epoch, 0
        torch.save({"state_dict": model.state_dict(), "mu": MU, "sigma": SIGMA,
                    "epoch": epoch, "val_metrics": va_m},
                   CKPT_DIR / "resnet50_best.pt")
        marker = "  ← mejor (guardado)"
    else:
        no_improve += 1
    print(f"[{epoch:02d}] train={tr_loss:.4f} val={va_loss:.4f} "
          f"MAE={va_m['mae_m']:6.1f} m  RMSE={va_m['rmse_m']:6.1f}  "
          f"F1={va_m['f1_macro']:.3f}  FS={va_m['false_safe_rate']:.3f}{marker}")
    scheduler.step()
    if no_improve >= PATIENCE:
        print(f"Early stopping en época {epoch} (mejor: {best_epoch})")
        break
print(f"\\nEntrenamiento: {(time.time()-t0)/60:.1f} min · mejor época {best_epoch} "
      f"(val MAE {best_mae:.1f} m)")

In [ ]:
# Curvas de aprendizaje
hist = pd.DataFrame(history)
fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
axes[0].plot(hist["epoch"], hist["train_loss"], "o-", label="train")
axes[0].plot(hist["epoch"], hist["val_loss"], "s-", label="val")
axes[0].set_xlabel("época"); axes[0].set_ylabel("MSE (y_norm)"); axes[0].legend()
axes[0].set_title("Pérdida")
axes[1].plot(hist["epoch"], hist["mae_m"], "o-", color="#b2182b")
axes[1].set_xlabel("época"); axes[1].set_ylabel("MAE (m)")
axes[1].set_title("MAE en validación")
fig.suptitle("ResNet-50 — curvas de aprendizaje")
plt.savefig(FIG_DIR / "resnet50_curvas.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Evaluación FINAL en test con el mejor checkpoint
ckpt = torch.load(CKPT_DIR / "resnet50_best.pt", map_location=DEVICE)
model.load_state_dict(ckpt["state_dict"])
with torch.no_grad():
    _, te_pred, te_true = run_epoch(model, test_loader)
resnet_metrics = compute_metrics(te_true, te_pred)
resnet_metrics.update({"model": "resnet50_baseline", "n_params_M": round(n_params/1e6, 1),
                       "best_epoch": best_epoch,
                       "train_time_min": round((time.time()-t0)/60, 1)})
print(json.dumps(resnet_metrics, indent=2))

In [ ]:
# Matriz de confusión (bandas derivadas de V̂) + F1 por banda
from sklearn.metrics import classification_report, confusion_matrix

b_true = np.digitize(te_true, BAND_EDGES)
b_pred = np.digitize(te_pred, BAND_EDGES)
cm = confusion_matrix(b_true, b_pred, labels=[0, 1, 2, 3])
fig, ax = plt.subplots(figsize=(5.5, 4.5), constrained_layout=True)
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(4)); ax.set_yticks(range(4))
ax.set_xticklabels(BAND_NAMES, rotation=25, ha="right")
ax.set_yticklabels(BAND_NAMES)
ax.set_xlabel("Predicha"); ax.set_ylabel("Real"); ax.set_title("ResNet-50 · test")
for i in range(4):
    for j in range(4):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max()/2 else "black")
plt.savefig(FIG_DIR / "resnet50_confusion.png", dpi=150, bbox_inches="tight")
plt.show()
print(classification_report(b_true, b_pred, target_names=BAND_NAMES, zero_division=0))

In [ ]:
# Exportamos métricas (Drive + repo local) para el notebook 06 y los slides
with open(RESULTS_DIR / "resnet50_metrics.json", "w") as f:
    json.dump(resnet_metrics, f, indent=2)

REPO_LOCAL = Path("/content/camanchaca-predict")
if REPO_LOCAL.exists():
    import shutil
    (REPO_LOCAL / "reports" / "figures").mkdir(parents=True, exist_ok=True)
    for fig in ("resnet50_curvas.png", "resnet50_confusion.png"):
        shutil.copy(FIG_DIR / fig, REPO_LOCAL / "reports" / "figures" / fig)
    (REPO_LOCAL / "reports" / "tables").mkdir(parents=True, exist_ok=True)
    shutil.copy(RESULTS_DIR / "resnet50_metrics.json",
                REPO_LOCAL / "reports" / "tables" / "resnet50_metrics.json")
print("Métricas y figuras exportadas ✅")

### Checklist de interpretación (para la presentación)

- ¿La curva de val sigue a la de train? (sobreajuste → R8)
- ¿El MAE en metros es aceptable para la banda donde más falla?
- **Falso-seguro**: ¿cuántos casos con V real < 100 m se predijeron ≥ 100 m?
  Es el número más importante para seguridad vial.
- La matriz de confusión muestra ¿dónde se concentran los errores? (típico:
  bandas adyacentes, que es el error "menos grave").

**Siguiente:** notebook 05 · vit_finetune — el modelo protagonista con
cabeza dual (regresión + bandas).

# Sección 5 · Modelo protagonista — ViT-B/16 con cabeza dual

**Fase A** (tronco congelado, solo cabezas) → **Fase B** (fine-tuning con LR
diferenciales). Pérdida conjunta `λ·MSE + (1−λ)·CE` con class weights.
La atención global de parches es la hipótesis del proyecto: la camanchaca es
**heterogénea**, y un CNN local puede no capturar su estructura global.

**Salida:** `checkpoints/vitb16_best.pt` + `results/vitb16_metrics.json`.

In [ ]:
# Datos: mismos splits, mismas transformaciones que el baseline
train_df = pd.read_csv(DATA_DIR / "splits" / "train.csv")
val_df = pd.read_csv(DATA_DIR / "splits" / "val.csv")
test_df = pd.read_csv(DATA_DIR / "splits" / "test.csv")
with open(DATA_DIR / "splits" / "class_weights.json") as f:
    stats = json.load(f)
MU, SIGMA = stats["mu"], stats["sigma"]
CLASS_WEIGHTS = torch.tensor(
    [stats["class_weights"][b] for b in
     ("critico", "alto_riesgo", "precaucion", "aceptable")],
    dtype=torch.float32).to(DEVICE)
print(f"train={len(train_df)} val={len(val_df)} test={len(test_df)}")

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
train_tf = transforms.Compose([
    transforms.Resize(256), transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(), transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
eval_tf = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(224),
    transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])

class VisibilityDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        img = self.transform(Image.open(r["path"]).convert("RGB"))
        return (img, float(r["y_norm"]), int(r["band_idx"]), float(r["visibility_m"]))

BATCH = 32          # <-- bajar a 16 si aparece CUDA OOM
train_loader = DataLoader(VisibilityDataset(train_df, train_tf), batch_size=BATCH,
                          shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(VisibilityDataset(val_df, eval_tf), batch_size=BATCH,
                        num_workers=2, pin_memory=True)
test_loader = DataLoader(VisibilityDataset(test_df, eval_tf), batch_size=BATCH,
                         num_workers=2, pin_memory=True)
print(f"Loaders listos · batch={BATCH}")

In [ ]:
# Modelo ViT-B/16 con cabeza dual (idéntico a src/camanchaca/models/)
class ViTVisibility(nn.Module):
    def __init__(self, n_bands=4, pretrained=True):
        super().__init__()
        self.trunk = timm.create_model(
            "vit_base_patch16_224.augreg_in21k_ft_in1k",
            pretrained=pretrained, num_classes=0)
        d = self.trunk.num_features
        self.head_reg = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, 1))
        self.head_cls = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, n_bands))
    def forward(self, x):
        f = self.trunk(x)
        return self.head_reg(f).squeeze(-1), self.head_cls(f)

model = ViTVisibility(pretrained=True).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"ViT-B/16 · {n_params/1e6:.1f} M parámetros")

In [ ]:
# Pérdida conjunta + métricas (mismas que el baseline para comparar justo)
LAM = 0.6   # peso de la regresión frente a la clasificación

def dual_loss(out_reg, out_cls, y_norm, band_idx):
    mse = F.mse_loss(out_reg, y_norm)
    ce = F.cross_entropy(out_cls, band_idx, weight=CLASS_WEIGHTS)
    return LAM * mse + (1 - LAM) * ce, mse, ce

BAND_EDGES = (50.0, 100.0, 200.0)
BAND_NAMES = ("critico", "alto_riesgo", "precaucion", "aceptable")

def to_meters(y_norm):
    return 10 ** (y_norm * SIGMA + MU)

def compute_metrics(v_true, v_pred):
    from sklearn.metrics import f1_score
    err = v_pred - v_true
    mae = float(np.abs(err).mean())
    rmse = float(np.sqrt((err ** 2).mean()))
    mape = float((np.abs(err) / np.clip(v_true, 1e-6, None)).mean() * 100)
    ss_res = float((err ** 2).sum())
    ss_tot = float(((v_true - v_true.mean()) ** 2).sum())
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else float("nan")
    b_true = np.digitize(v_true, BAND_EDGES)
    b_pred = np.digitize(v_pred, BAND_EDGES)
    f1 = float(f1_score(b_true, b_pred, average="macro", zero_division=0))
    peligro = v_true < 100.0
    falso_seguro = float((v_pred[peligro] >= 100.0).mean()) if peligro.any() else 0.0
    return {"mae_m": mae, "rmse_m": rmse, "mape_pct": mape, "r2": r2,
            "f1_macro": f1, "false_safe_rate": falso_seguro}

def run_epoch(model, loader, optimizer=None, scaler=None):
    training = optimizer is not None
    model.train() if training else model.eval()
    sum_loss, preds, vtrues = 0.0, [], []
    with torch.set_grad_enabled(training):
        for x, y_norm, band, v_true in loader:
            x = x.to(DEVICE, non_blocking=True)
            y_norm = y_norm.to(DEVICE, non_blocking=True)
            band = band.to(DEVICE, non_blocking=True)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16,
                                enabled=training and scaler is not None):
                out_reg, out_cls = model(x)
                loss, mse, ce = dual_loss(out_reg.float(), out_cls.float(), y_norm, band)
            if training:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimizer); scaler.update()
            sum_loss += loss.item() * x.size(0)
            preds.append(out_reg.float().detach().cpu().numpy())
            vtrues.append(v_true.numpy())
    return (sum_loss / len(loader.dataset),
            to_meters(np.concatenate(preds)),
            np.concatenate(vtrues))

### Fase A — warmup de las cabezas (tronco congelado)

Evita que gradientes grandes de cabezas aleatorias destruyan los pesos
preentrenados del tronco. Es rápido (solo se actualizan ~0.2 M parámetros).

In [ ]:
# FASE A: congelamos el tronco y entrenamos solo las cabezas
for p in model.trunk.parameters():
    p.requires_grad = False

EPOCHS_A = 1 if FAST_RUN else 2
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad], lr=1e-3)
scaler = torch.amp.GradScaler(enabled=DEVICE.type == "cuda")
history = []

t0 = time.time()
for epoch in range(1, EPOCHS_A + 1):
    tr_loss, _, _ = run_epoch(model, train_loader, optimizer, scaler)
    with torch.no_grad():
        va_loss, va_pred, va_true = run_epoch(model, val_loader)
    m = compute_metrics(va_true, va_pred)
    history.append({"fase": "A", "epoch": epoch, "train_loss": tr_loss,
                    "val_loss": va_loss, **m})
    print(f"[A{epoch}] train={tr_loss:.4f} val={va_loss:.4f} MAE={m['mae_m']:6.1f} m "
          f"F1={m['f1_macro']:.3f} FS={m['false_safe_rate']:.3f}")

### Fase B — fine-tuning completo con LR diferenciales

El tronco aprende con LR bajo (1e-5, preserva el preentrenamiento 21k) y las
cabezas con LR alto (1e-4). Early stopping sobre MAE de validación;
checkpoint por mejora en Drive (mitigación R4/R5).

In [ ]:
# FASE B: descongelamos todo y entrenamos con LR diferenciales
for p in model.trunk.parameters():
    p.requires_grad = True

EPOCHS_B = 2 if FAST_RUN else 6
PATIENCE = 3
optimizer = torch.optim.AdamW([
    {"params": model.trunk.parameters(), "lr": 1e-5},
    {"params": model.head_reg.parameters(), "lr": 1e-4},
    {"params": model.head_cls.parameters(), "lr": 1e-4},
], weight_decay=0.05)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_B)

best_mae, best_epoch, no_improve = float("inf"), -1, 0
for epoch in range(1, EPOCHS_B + 1):
    tr_loss, _, _ = run_epoch(model, train_loader, optimizer, scaler)
    with torch.no_grad():
        va_loss, va_pred, va_true = run_epoch(model, val_loader)
    m = compute_metrics(va_true, va_pred)
    history.append({"fase": "B", "epoch": EPOCHS_A + epoch, "train_loss": tr_loss,
                    "val_loss": va_loss, **m})
    marker = ""
    if m["mae_m"] < best_mae:
        best_mae, best_epoch, no_improve = m["mae_m"], EPOCHS_A + epoch, 0
        torch.save({"state_dict": model.state_dict(), "mu": MU, "sigma": SIGMA,
                    "lam": LAM, "epoch": EPOCHS_A + epoch, "val_metrics": m},
                   CKPT_DIR / "vitb16_best.pt")
        marker = "  ← mejor (guardado)"
    else:
        no_improve += 1
    print(f"[B{epoch}] train={tr_loss:.4f} val={va_loss:.4f} MAE={m['mae_m']:6.1f} m "
          f"F1={m['f1_macro']:.3f} FS={m['false_safe_rate']:.3f}{marker}")
    scheduler.step()
    if no_improve >= PATIENCE:
        print(f"Early stopping en B{epoch} (mejor: época {best_epoch})")
        break
print(f"\\nTiempo total: {(time.time()-t0)/60:.1f} min · mejor época {best_epoch} "
      f"(val MAE {best_mae:.1f} m)")

In [ ]:
# Curvas de aprendizaje (fases A + B)
hist = pd.DataFrame(history)
fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
axes[0].plot(hist["epoch"], hist["train_loss"], "o-", label="train")
axes[0].plot(hist["epoch"], hist["val_loss"], "s-", label="val")
axes[0].axvline(EPOCHS_A + 0.5, color="gray", ls=":", lw=1)
axes[0].text(EPOCHS_A + 0.6, axes[0].get_ylim()[1]*0.95, "fase B", color="gray")
axes[0].set_xlabel("época"); axes[0].set_ylabel("pérdida dual"); axes[0].legend()
axes[1].plot(hist["epoch"], hist["mae_m"], "o-", color="#b2182b")
axes[1].set_xlabel("época"); axes[1].set_ylabel("MAE (m)")
fig.suptitle("ViT-B/16 — curvas de aprendizaje (fase A: warmup, fase B: fine-tuning)")
plt.savefig(FIG_DIR / "vitb16_curvas.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Evaluación FINAL en test con el mejor checkpoint
ckpt = torch.load(CKPT_DIR / "vitb16_best.pt", map_location=DEVICE)
model.load_state_dict(ckpt["state_dict"])
with torch.no_grad():
    _, te_pred, te_true = run_epoch(model, test_loader)
vit_metrics = compute_metrics(te_true, te_pred)
vit_metrics.update({"model": "vit_b16_dual", "n_params_M": round(n_params/1e6, 1),
                    "best_epoch": best_epoch, "lam_reg": LAM,
                    "train_time_min": round((time.time()-t0)/60, 1)})
print(json.dumps(vit_metrics, indent=2))

In [ ]:
# Matriz de confusión + reporte por banda (cabeza de clasificación implícita)
from sklearn.metrics import classification_report, confusion_matrix

b_true = np.digitize(te_true, BAND_EDGES)
b_pred = np.digitize(te_pred, BAND_EDGES)
cm = confusion_matrix(b_true, b_pred, labels=[0, 1, 2, 3])
fig, ax = plt.subplots(figsize=(5.5, 4.5), constrained_layout=True)
ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(4)); ax.set_yticks(range(4))
ax.set_xticklabels(BAND_NAMES, rotation=25, ha="right")
ax.set_yticklabels(BAND_NAMES)
ax.set_xlabel("Predicha"); ax.set_ylabel("Real"); ax.set_title("ViT-B/16 · test")
for i in range(4):
    for j in range(4):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max()/2 else "black")
plt.savefig(FIG_DIR / "vitb16_confusion.png", dpi=150, bbox_inches="tight")
plt.show()
print(classification_report(b_true, b_pred, target_names=BAND_NAMES, zero_division=0))

In [ ]:
# Exportamos métricas y figuras (Drive + repo local)
with open(RESULTS_DIR / "vitb16_metrics.json", "w") as f:
    json.dump(vit_metrics, f, indent=2)

REPO_LOCAL = Path("/content/camanchaca-predict")
if REPO_LOCAL.exists():
    import shutil
    (REPO_LOCAL / "reports" / "figures").mkdir(parents=True, exist_ok=True)
    for fig in ("vitb16_curvas.png", "vitb16_confusion.png"):
        shutil.copy(FIG_DIR / fig, REPO_LOCAL / "reports" / "figures" / fig)
    (REPO_LOCAL / "reports" / "tables").mkdir(parents=True, exist_ok=True)
    shutil.copy(RESULTS_DIR / "vitb16_metrics.json",
                REPO_LOCAL / "reports" / "tables" / "vitb16_metrics.json")
print("Métricas y figuras exportadas ✅")

### Checklist de interpretación

- ¿El ViT supera al ResNet-50 en MAE y en falso-seguro? (ese es el argumento
  central del proyecto)
- ¿La fase A estabilizó el inicio del fine-tuning? (comparar con un run sin
  warmup sería un ablation interesante para la entrega)
- Sobreajuste: si train baja y val sube en fase B → subir weight decay o
  congelar más bloques (R8).

**Siguiente:** notebook 06 · eval_compare — comparación final ResNet vs ViT,
figuras para la presentación y test cualitativo O-HAZE.

# Sección 6 · Evaluación comparativa — ResNet-50 vs ViT-B/16 (+ O-HAZE)

Mismo test set para ambos modelos: métricas de regresión (MAE/RMSE/MAPE/R²),
de clasificación (F1 macro, matriz de confusión), **tasa de falso-seguro**,
latencia y figuras listas para los slides. O-HAZE (niebla real) es opcional.

**Salida:** `figures/*.png` + `results/resumen_para_slides.md`.

In [ ]:
# Reconstruimos ambos modelos (idénticos a notebooks 04/05) y cargamos pesos
class ResNetVisibility(nn.Module):
    def __init__(self):
        super().__init__()
        net = models.resnet50(weights=None)
        net.fc = nn.Identity()
        self.backbone = net
        self.head = nn.Sequential(nn.Linear(2048, 256), nn.ReLU(inplace=True),
                                  nn.Dropout(0.2), nn.Linear(256, 1))
    def forward(self, x):
        return self.head(self.backbone(x)).squeeze(-1)

class ViTVisibility(nn.Module):
    def __init__(self, n_bands=4):
        super().__init__()
        self.trunk = timm.create_model(
            "vit_base_patch16_224.augreg_in21k_ft_in1k", pretrained=False,
            num_classes=0)
        d = self.trunk.num_features
        self.head_reg = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, 1))
        self.head_cls = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, n_bands))
    def forward(self, x):
        f = self.trunk(x)
        return self.head_reg(f).squeeze(-1), self.head_cls(f)

resnet = ResNetVisibility().to(DEVICE)
resnet.load_state_dict(torch.load(CKPT_DIR / "resnet50_best.pt",
                                  map_location=DEVICE)["state_dict"])
vit = ViTVisibility().to(DEVICE)
vit.load_state_dict(torch.load(CKPT_DIR / "vitb16_best.pt",
                               map_location=DEVICE)["state_dict"])
with open(DATA_DIR / "splits" / "class_weights.json") as f:
    stats = json.load(f)
MU, SIGMA = stats["mu"], stats["sigma"]
print("Checkpoints cargados ✅")

In [ ]:
# Test set + loaders (idénticos a los notebooks de entrenamiento)
test_df = pd.read_csv(DATA_DIR / "splits" / "test.csv")
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
eval_tf = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(224), transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])

class VisibilityDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        img = self.transform(Image.open(r["path"]).convert("RGB"))
        return (img, float(r["y_norm"]), int(r["band_idx"]), float(r["visibility_m"]))

test_loader = DataLoader(VisibilityDataset(test_df, eval_tf), batch_size=64,
                         num_workers=2, pin_memory=True)
print(f"test: {len(test_df)} imágenes")

In [ ]:
# Predicciones de ambos modelos sobre el mismo test
@torch.no_grad()
def predict(model, loader, dual=False):
    model.eval()
    preds, vtrues = [], []
    for x, y_norm, band, v_true in loader:
        x = x.to(DEVICE, non_blocking=True)
        out = model(x)
        reg = out[0] if dual else out
        preds.append(reg.float().cpu().numpy())
        vtrues.append(v_true.numpy())
    return np.concatenate(preds), np.concatenate(vtrues)

y_rn, v_true = predict(resnet, test_loader, dual=False)
y_vt, _ = predict(vit, test_loader, dual=True)

def to_meters(y_norm):
    return 10 ** (y_norm * SIGMA + MU)
v_pred_rn = to_meters(y_rn)
v_pred_vt = to_meters(y_vt)
print(f"Predicciones listas: ResNet {len(v_pred_rn)} · ViT {len(v_pred_vt)}")

In [ ]:
# Métricas + latencia (ms/imagen en la GPU actual)
def compute_metrics(v_true, v_pred):
    from sklearn.metrics import f1_score
    err = v_pred - v_true
    mae = float(np.abs(err).mean())
    rmse = float(np.sqrt((err ** 2).mean()))
    mape = float((np.abs(err) / np.clip(v_true, 1e-6, None)).mean() * 100)
    ss_res = float((err ** 2).sum())
    ss_tot = float(((v_true - v_true.mean()) ** 2).sum())
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else float("nan")
    f1 = float(f1_score(np.digitize(v_true, (50, 100, 200)),
                        np.digitize(v_pred, (50, 100, 200)),
                        average="macro", zero_division=0))
    peligro = v_true < 100.0
    fs = float((v_pred[peligro] >= 100.0).mean()) if peligro.any() else 0.0
    return {"mae_m": mae, "rmse_m": rmse, "mape_pct": mape, "r2": r2,
            "f1_macro": f1, "false_safe_rate": fs}

def latency_ms(model, dual=False, n_warmup=10, n_iter=50):
    model.eval()
    dummy = torch.randn(1, 3, 224, 224, device=DEVICE)
    with torch.no_grad():
        for _ in range(n_warmup):
            model(dummy)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(n_iter):
            model(dummy)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
    return (time.perf_counter() - t0) / n_iter * 1000

m_rn = compute_metrics(v_true, v_pred_rn)
m_vt = compute_metrics(v_true, v_pred_vt)
m_rn.update({"model": "ResNet-50 (baseline)", "latency_ms": round(latency_ms(resnet), 1)})
m_vt.update({"model": "ViT-B/16 (protagonista)", "latency_ms": round(latency_ms(vit, dual=True), 1)})

tabla = pd.DataFrame([m_rn, m_vt]).set_index("model")
tabla[["mae_m", "rmse_m", "mape_pct", "r2", "f1_macro",
       "false_safe_rate", "latency_ms"]].round(3)

In [ ]:
# FIGURA (slides): scatter pred vs real, log-log, ambos modelos
fig, axes = plt.subplots(1, 2, figsize=(11, 4.6), constrained_layout=True)
for ax, (nombre, vp) in zip(axes, [("ResNet-50", v_pred_rn), ("ViT-B/16", v_pred_vt)]):
    ax.scatter(v_true, vp, s=10, alpha=0.45, color="#3681a6", edgecolors="none")
    lims = [max(5, v_true.min()), max(v_true.max(), vp.max())]
    ax.plot(lims, lims, "r--", lw=1.2, label="y = x")
    for e in (50, 100, 200):
        ax.axvline(e, color="gray", ls=":", lw=0.8)
        ax.axhline(e, color="gray", ls=":", lw=0.8)
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel("Visibilidad real [m]"); ax.set_ylabel("Visibilidad predicha [m]")
    ax.set_title(nombre); ax.legend()
fig.suptitle("Test: predicción vs realidad (líneas grises = cortes de banda)")
plt.savefig(FIG_DIR / "scatter_pred_vs_true.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# FIGURA (slides): error absoluto por banda real
BAND_NAMES = ("critico", "alto_riesgo", "precaucion", "aceptable")
b_true = np.digitize(v_true, (50, 100, 200))
err_rn = np.abs(v_pred_rn - v_true)
err_vt = np.abs(v_pred_vt - v_true)

fig, ax = plt.subplots(figsize=(8, 4.2), constrained_layout=True)
data_rn = [err_rn[b_true == i] for i in range(4)]
data_vt = [err_vt[b_true == i] for i in range(4)]
pos = np.arange(4)
bp1 = ax.boxplot(data_rn, positions=pos - 0.19, widths=0.34, patch_artist=True,
                 medianprops=dict(color="k"))
bp2 = ax.boxplot(data_vt, positions=pos + 0.19, widths=0.34, patch_artist=True,
                 medianprops=dict(color="k"))
for b in bp1["boxes"]:
    b.set_facecolor("#67a9cf")
for b in bp2["boxes"]:
    b.set_facecolor("#2166ac")
ax.set_xticks(pos); ax.set_xticklabels(BAND_NAMES)
ax.set_ylabel("Error absoluto [m]")
ax.legend([bp1["boxes"][0], bp2["boxes"][0]], ["ResNet-50", "ViT-B/16"])
ax.set_title("Error absoluto por banda real (test)")
plt.savefig(FIG_DIR / "error_por_banda.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# FIGURA (slides): matrices de confusión lado a lado
from sklearn.metrics import confusion_matrix

fig, axes = plt.subplots(1, 2, figsize=(11, 4.6), constrained_layout=True)
for ax, (nombre, vp) in zip(axes, [("ResNet-50", v_pred_rn), ("ViT-B/16", v_pred_vt)]):
    cm = confusion_matrix(b_true, np.digitize(vp, (50, 100, 200)), labels=[0, 1, 2, 3])
    ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(4)); ax.set_yticks(range(4))
    ax.set_xticklabels(BAND_NAMES, rotation=25, ha="right")
    ax.set_yticklabels(BAND_NAMES)
    ax.set_xlabel("Predicha"); ax.set_ylabel("Real"); ax.set_title(nombre)
    for i in range(4):
        for j in range(4):
            ax.text(j, i, cm[i, j], ha="center", va="center",
                    color="white" if cm[i, j] > cm.max()/2 else "black")
plt.savefig(FIG_DIR / "confusion_comparada.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# FIGURA (slides): barras comparativas (MAE y falso-seguro)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), constrained_layout=True)
nombres = ["ResNet-50", "ViT-B/16"]
axes[0].bar(nombres, [m_rn["mae_m"], m_vt["mae_m"]],
            color=["#67a9cf", "#2166ac"], width=0.55)
axes[0].set_ylabel("MAE (m)"); axes[0].set_title("Error absoluto medio")
for i, v in enumerate([m_rn["mae_m"], m_vt["mae_m"]]):
    axes[0].text(i, v, f"{v:.1f} m", ha="center", va="bottom")

axes[1].bar(nombres, [m_rn["false_safe_rate"], m_vt["false_safe_rate"]],
            color=["#67a9cf", "#2166ac"], width=0.55)
axes[1].set_ylabel("tasa de falso-seguro"); axes[1].set_title("V real < 100 m predicha ≥ 100 m")
for i, v in enumerate([m_rn["false_safe_rate"], m_vt["false_safe_rate"]]):
    axes[1].text(i, v, f"{v:.1%}", ha="center", va="bottom")
plt.savefig(FIG_DIR / "comparativa_barras.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Ejemplos cualitativos: mejores y peores predicciones del ViT
err_vt_abs = err_vt
orden = np.argsort(err_vt_abs)
mejores, peores = orden[:4], orden[-4:]

fig, axes = plt.subplots(2, 4, figsize=(14, 6.6), constrained_layout=True)
for fila, (idxs, titulo) in enumerate([(mejores, "Mejores predicciones"),
                                       (peores, "Peores predicciones")]):
    for k, idx in enumerate(idxs):
        ax = axes[fila, k]
        r = test_df.iloc[idx]
        ax.imshow(Image.open(r["path"]))
        ax.axis("off")
        ax.set_title(f"real {v_true[idx]:.0f} m · ViT {v_pred_vt[idx]:.0f} m\n"
                     f"({BAND_NAMES[b_true[idx]]})", fontsize=9)
    axes[fila, 0].text(-0.06, 0.5, titulo, transform=axes[fila, 0].transAxes,
                       rotation=90, va="center", ha="center", fontsize=11)
plt.savefig(FIG_DIR / "cualitativos_vit.png", dpi=150, bbox_inches="tight")
plt.show()

### Test cualitativo O-HAZE (niebla REAL) — riesgo R1

O-HAZE no tiene etiqueta en metros: aquí **no medimos, inspeccionamos**. Si
los modelos predicen visibilidades plausibles (bajas) y ordenan las escenas
de forma consistente con su densidad aparente de niebla, hay evidencia de
generalización síntesis → real. Si no, documentamos la brecha (honestidad
científica → `docs/risks_and_mitigation.md` R1).

> O-HAZE se solicita con correo institucional en la página del NTIRE 2018
> Image Dehazing Challenge y se organiza con
> `python scripts/download_ohaze.py --zip O-HAZE.zip`
> (carpeta esperada: `data/raw/ohaze/hazy/`).

In [ ]:
# O-HAZE (condicional): inferencia cualitativa sin etiqueta
OHAZE_DIR = DATA_DIR / "raw" / "ohaze" / "hazy"
hazy_paths = sorted(OHAZE_DIR.glob("*.png")) + sorted(OHAZE_DIR.glob("*.jpg")) \
    if OHAZE_DIR.exists() else []

if not hazy_paths:
    print("O-HAZE no disponible — el test cualitativo se omite.")
    print("Instrucciones en la celda markdown anterior y en scripts/download_ohaze.py")
else:
    @torch.no_grad()
    def predict_single(model, pil_img, dual=False):
        x = eval_tf(pil_img.convert("RGB")).unsqueeze(0).to(DEVICE)
        out = model(x)
        reg = out[0] if dual else out
        return float(to_meters(reg.float().cpu().numpy()[0]))

    sel = hazy_paths[:8]
    fig, axes = plt.subplots(2, 4, figsize=(14, 6.6), constrained_layout=True)
    for k, p in enumerate(sel):
        ax = axes[k // 4, k % 4]
        img = Image.open(p)
        vr = predict_single(resnet, img)
        vv = predict_single(vit, img, dual=True)
        banda = BAND_NAMES[min(3, np.digitize(vv, (50, 100, 200)))]
        ax.imshow(img); ax.axis("off")
        ax.set_title(f"ResNet {vr:.0f} m · ViT {vv:.0f} m\n({banda})",
                     fontsize=9, color="#8b0000")
    fig.suptitle("O-HAZE (niebla REAL, sin etiqueta): predicciones de visibilidad", fontsize=13)
    plt.savefig(FIG_DIR / "ohaze_cualitativo.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
# Exportamos TODO a reports/ (repo) + resumen para los slides
import shutil

REPO_LOCAL = Path("/content/camanchaca-predict")
figuras = ["scatter_pred_vs_true.png", "error_por_banda.png",
           "confusion_comparada.png", "comparativa_barras.png",
           "cualitativos_vit.png"]
if OHAZE_DIR.exists() and hazy_paths:
    figuras.append("ohaze_cualitativo.png")

filas = ["| Modelo | MAE (m) | RMSE (m) | MAPE | R2 | F1 macro | Falso-seguro | Latencia (ms) |",
         "|---|---|---|---|---|---|---|---|"]
for m in (m_rn, m_vt):
    filas.append(f"| {m['model']} | {m['mae_m']:.1f} | {m['rmse_m']:.1f} | "
                 f"{m['mape_pct']:.1f}% | {m['r2']:.3f} | {m['f1_macro']:.3f} | "
                 f"{m['false_safe_rate']:.1%} | {m['latency_ms']:.1f} |")
resumen = "# Resultados para la presentación (generado por el notebook 06)\n\n" + "\n".join(filas) + "\n"
(RESULTS_DIR / "resumen_para_slides.md").write_text(resumen, encoding="utf-8")
print(resumen)

if REPO_LOCAL.exists():
    figdir = REPO_LOCAL / "reports" / "figures"
    figdir.mkdir(parents=True, exist_ok=True)
    for f in figuras:
        if (FIG_DIR / f).exists():
            shutil.copy(FIG_DIR / f, figdir / f)
    shutil.copy(RESULTS_DIR / "resumen_para_slides.md",
                REPO_LOCAL / "reports" / "tables" / "resumen_para_slides.md")
    print(f"Figuras copiadas al repo: {len(figuras)}")

with open(RESULTS_DIR / "comparison.json", "w") as f:
    json.dump({"resnet": m_rn, "vit": m_vt}, f, indent=2)
print("comparison.json guardado ✅")

### Conclusiones (plantilla para completar tras el run)

1. **Comparación:** ¿el ViT superó al ResNet-50 en MAE y falso-seguro?
   ¿A qué costo (parámetros ×3.5, latencia ×?)?
2. **Seguridad:** la tasa de falso-seguro es la métrica de despliegue; un
   sistema real exigiría umbral conservador (predicción mínima de un ensemble).
3. **Generalización (O-HAZE):** comportamiento cualitativo en niebla real y
   brecha documentada (R1).
4. **Camanchaca real:** sin datos chilenos en esta fase — roadmap de capturas
   propias/cámaras DTV para fine-tuning local.

Con estas figuras y `resumen_para_slides.md` tienes TODO el material
cuantitativo de la presentación. El deck sigue el guion de
`docs/avance_2026-09-21.md`.

## ✅ Fin del pipeline — ¿dónde quedaron los resultados?

| Artefacto | Ruta (en Google Drive) |
|---|---|
| Manifiesto de datos | `camanchaca/data/processed/reside_manifest.csv` |
| Etiquetas en metros | `camanchaca/data/processed/labels.csv` |
| Splits train/val/test | `camanchaca/data/splits/` |
| Checkpoints | `camanchaca/checkpoints/` |
| Métricas JSON | `camanchaca/results/` |
| Figuras para slides | `camanchaca/figures/` |

Con `FAST_RUN = True` los números son de **demo** (set sintético pequeño): sirven
para verificar el flujo completo y ensayar la presentación. Para los resultados
reales de la entrega: `FAST_RUN = False` + `kaggle.json` en Drive +
`DATASET_SLUG` verificado, y ejecutar todo de nuevo.